[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ratchanon12/2306565/blob/main/boiler_intro.ipynb)

# What is a database? — hands-on notebook

**One month of data from a boiler house · April 2026**

Something in this boiler house is wrong. Nobody has told you what it is, and no single row of
data will show you. By the end of the notebook you will have found it, worked out how fast it
is getting worse, and backed it up with two independent sets of numbers.

You do not need any experience with databases. Run the cells from the top. Where you see
**`# YOUR TURN`** there is a query with **`___`** blanks in it — fill them in and run the cell.
You cannot break anything.

> ### Working in Google Colab? Do this first
> **File → Save a copy in Drive.** That gives you your own copy, saved to your Google
> account, so the answers you type today are still there tomorrow. If you skip it, your work
> disappears when you close the tab.
>
> Run cells with the ▶ button on the left of each cell, or **Shift + Enter**.

### The five tables

| Table | One row is | Rows |
|---|---|---|
| `boilers` | one boiler — the asset register | 3 |
| `operators` | one person on the shift roster | 6 |
| `readings` | one instrument reading, every 4 hours | 540 |
| `daily_logs` | one boiler on one day | 90 |
| `water_tests` | one weekly water sample | 15 |

### The shape of every query you will write today

```
WITH     a named sub-query                      (Exercise 5 only)
SELECT   which columns you want
FROM     which table
JOIN     another table  ON  how they match      (only when you need it)
WHERE    which rows you want                    (optional)
GROUP BY what to summarise by                   (optional)
HAVING   which groups to keep                   (optional)
ORDER BY how to sort                            (optional)
LIMIT    how many rows to show                  (optional)
```

Only `SELECT` and `FROM` are compulsory. The clauses must appear in that order.

### How hard is this?

| | |
|---|---|
| Exercises 1–3 | the basics — fill in one or two blanks |
| Exercise 4 | sharper questions — `CASE`, sub-queries, dates. Fewer blanks are filled in for you |
| Exercise 5 | one query that makes the whole case. Mostly you |
| Take it further | no answers provided |

## Step 1 — set the database up

Run the next two cells. The first one **builds the database for you** — there is nothing to
upload and nothing to install. The second one connects to it and gives you three tools.

> If Colab disconnects or you come back tomorrow, just run these two cells again.

In [ ]:
# ---- RUN ME FIRST -------------------------------------------------------
# This builds boilerhouse.db. You do not need to read or understand this cell.
import base64, gzip, sqlite3, os

_DATA = """
H4sIAKDolmoC/7VdW4+ct5F9969oBNiVA4xmyWLxljzJyiTRRpINWdnET0Jr1JZ6PRdhpide//vl5buwqsfuowVWQAJb7j7N
w0uxWMU6/ObiLy9eb96+efb6+2fP37749vUfv3r+5uLZ24vN22ffvLzYvL/dX+3u7jdff7Upf/q/vdt/qP/y4vXbi79cvNl8
9+bFq2dvftj87eKHs/apy9sPu8385+3FP99uXn9b/vf3ly/P+t89fbo5bD9ubh6u3+/uNrc3m8On3ea7f3/xp/b1m+31ya//
/Gl7aN+6/by72x5uSwsvt1dXm/2hQVxvfyrNvC7tuNIQ7b+Xr+xvPr47fP5U/qXQffnIT3zY3e8/3mzuD7vt9eb24fD54XC2
Odze3Ow25Tc3n24f7hpW/9y799u7j6ewPt/t7u8f7nZnm/Lpzcftw8fd1GHX1/v7+/3tze7D0q0jSoH4ZVe+sj9sdjeH3V35
2P3u7l/7y/79Hx8qz8d67Kvf//GrF6+/v3jztuJ+u/ndNJ6/2/zXs5d/v/j+a3v25JunZI19cvbk9e3d4dPm5235gcPD+135
m2+27y9vL3/a/PnVU0vmyZnz5+aMqfwfGZPLN7aHh7vtVaFy/+TUL9H0S67gfl/68//vl9z0S1Rwn5f+utuJn3q1P9w/vN/f
f9pvXn3zNNcfM+uP2XD0Y2JBfNjur355d3X7cV4T5R+nBfEbS2JcOHqAy6T588Wbi9fPL76fl9vXy+d/378/T/OK8FvfX5bD
18M3Joza0A+lH35jWT35ofx5+urV0z/96Un7Tpv97w7tv//K3O4L5Hpb1vyhrsoP21+mdbLMzXcf//sYYbMglG7evH+4uzkM
AH/5z6Opu/b8MHvtWZlXZCg8Nfy0zuKQzHku4xjp3AEIVL7vFIJz5+7M5pjPDYDgyvdZIvjoylyy0btzBhC4sBjbUOZtNK6y
sGVSIgi+sGCJEDiel84x5M4JQAilBV4hOFtZ5LI6EoAQC4uxDWWdh+zPQ2Fhw3kGEFJh4RUCh84Ca0MuLIJCsLGOZkr53CIz
qjAWjeBKI7bBKF2J0LB1TgYF4c15mWrG8blHIOqktBLCl870hQhhU8JWgLEVvq6NWMfDZDoPCAQXIlZBhHhe/tqU7oSI+EKE
FIStltamskKRMS0G2YpWhLI8bG4jki20RG08I9GKUGeWK91Z7L2H7IRNZ04s0gphTOkLm1yElpjNylrFOiLuPNbuNNDUIqPM
VYUIphEJ+TwiEFbZqwLhU7N4MTHWClIGKwmDhfQFOWWxUiVCbY1Ej7WClcmqENXS1MUeMQivbFauI0J9/0jQMqOgjFbuRKrR
igFaIxSV1cp1sds2tUzGiCRptaypRHybFylgI5Kl1WoQru6lbSdE+sIZabUahHVtREp3In3hrLRa1tbFntrUSgmy4I6k1aoQ
gU2Z4NWThLrTOWm1KoTPrS+SSVB3OpZWy9YNtQxqwbUG20ecl1arQaS22G0xOZBvEqTVahC2zs5mfqFWRGm1bN1Ty1DUTSBj
ttMlabUaBHExfDYHbBNwWVqtClHXSB0RazEvy0irZblOLWrdGRK0IbKVVqtChNislrVYdzJJq2Xbzh7OU93ZA9Sd7KTVsm1n
D/O2jMwLZmm1GgTbBmGxqcVeWq0K4WN1DmwMARpUDspq1Z29rK/qMvoA7WYcldUK3X+vViszNi+Sslqhr5E6ImUrQkwOZ2W1
4jIvDGO7mTfKasXugJcdKpfdDOlOb5XVqhDGNaeRse70pKxW6vtIm1ogEaesVoXwqW+I4GmGldVK3deqVssFDMIrq1W35bKN
Ve83YFuRD8pqVQjfjyPgBPdRWa0KYTuRgC0zn6TVqrGRUFy9OiKg3+mztFoNItrqd6Ins2Ck1WoQJjQ33jNktYKVVovqzl4g
2mI30IgEklarQgTqiz1gfRGctFpkha+FbAKBpdWiaWe3fWeHiHhptRpE6PGnhPkXIUirVSF8qrtZceMJa0WUVoucOI8gEzwk
abUaBNl5Z0dsZ8jSalHb2WNbI4QNajTSalHd2Yu9SNVRwrblaKXVqhCB6mIv3i+2UiNJq0VtZ6fuomBbUXTSapHvB6s6tRzm
ukaWVovamT01L6c4SlAgx0ur1SDKtlwXe8BWagzKaoW+RuqIGI/1RVRWq+3s1KZWxlZqTMpqhbpGUtvZjYU2xJiV1Yrd10rV
+zXQBE9GWa0KwS1+YYrtREYkWWW16pk95x58YGiNJFJWK/UoCvWDFRSec8pqtZ297SMmWGheJFZWK9URaYs9ZsxFSV5ZrVyJ
xDbBq/FCIIKyWrlHUVx3oJGplaKyWhWizCmqI5KxQU3SajnTp1bbBCJGJEur1SDKhpiq1cKmVjbSalWIOiKxjkh3lET6Y823
ff2riYmj7Mfu+vO7m9tfT+jVJMDn7S93t1dXU1JwSh1cXb2bsoGPJPHuP+1/PGx+G/XZ2eabze3d5nnPs+3uDu8ur7b39+ob
m/UbPfOy8Grf2f+4v9wedkcZiaUzxnTaxaunzNG2LNf15aftfvP93f7+4afyF89qQqr9vH1yGosalrc1sPXk9fZut9v84/bm
Y4UUWARgud6uUB23J89utjebb25vb653LfX2Ra3i3ipyNY/33f5687y055ft59u7G4GFtMr3VqUaFX7yt/3hsP9cCG5e3Bw+
bctHK9YXtS30tjku9vHJP7Y3+w/bzXef7m6vr7cCi46yene77Yf9zZLTm/5V5ut+M6n3f8rqHe43m9/KdMuM3Oavf/3Dq1ci
L9fy148l5pa83I9Xtz8/mra+P2wvf3p3qEvz8lcQfrx62LXkXP1U7euWuv6w+9hX0y29+3w5L8BHUnsjwu3//PJxd3O2+bej
NTT3vMjpDdm0jTF/MGXWUtu3yZf/Lw5RPAkjA/4rTAu1xHrugGBk0H+F8dVCUmowLpyEYU2KO0yuWx/5BkPuJIzXpHgmVdON
uexgBSadhAmaFM+kXGtN+ZnzzCdhoiaV5pEqezRxb40/CZM0qQkmNlKpD/hJlKw5pZkTLQN1GqWl/0YYSw2m3RCYZ5873cNW
5aVnHKoTr06/XJtzum+syk4vOFzPFZQaDp0eKus0rzCPVZ3HrncPncZhzSvMgxUbrwS2x2teQU7BhgOMV1C8yMwLq1qLeqTG
+jkqXrSYi7zyOm0uWprwMRw/zh+gn7PgRYvdaU4/1WCvO+fTOC1h+BhOrMuiJmUKjjndz6RuKAx2sLanevCln08bMCLNi+d5
SG0eemwektO8eOZVzXudRcWEnV6nLYn4GI6vhtnVpH0Z93wax2teixGr69R2nNPzsKUTH8OZ7Uas8wfAiZpXkvsWYeu0JRZH
nNn+5LZOY5uHSP9kxWvGaZczYjuHnufT+2hLMT6G49t6z709p8fdWc0rDPOnWNdqN9zp+exI81rsWG7ry2Lj5ZzmFWY7T20e
orxY8aLFe8rT1u7OAVpe0aLFbNS4nOnL/TRMUKxmGJ5Gq7k9p2Gko+EGl9B/iTF00tNwg08YV58Q6GTpa6w4fVOODYdPdw8b
zWsxht3IZ8w4s9W8FrfQt8XVnNTTmxeT5rUYQ9+MRp08dLqf2WleSXq7GfO9mTWvNG5e6PRhr2klSavuyaeXKAfFajWF1ExG
nYX2tEnlqFitptCsawtoT1K0VlPYTUaZ7ecJ6J6seS2uYXfFmuk5bZq90bzCYDNmU3gaxmpaYXTkY9u5PNAcUrRWz7APV8Wx
p3cu7xQt6Rm2bPV5BHBY8VptYWy2sHlip4fLS0+DB8/QLMYQ2Lm89DR48Ay7x+uxaeilp8GDMUzLCQU4IfukeS0eXTeGAfPo
fNa8RpzZkz/dP8FoXjx6UBHGsZrXYgx9W17gPAykeY0eZjEbbfM63c/BaV6jh1mvoNTxOm3GAites/mZ4zSVlzu9eQWveK3m
sLsapo776c0iBMVrPSnbZR4aoH+i5hXGTZn6CQ4Yr6R5hWEeJtv6GWlP1rzCeKKcxut0P0ejeE32Z4psuHay4NPzOVrFa3UN
wxSueSTlegxDitZ6UE5teTUfAaAlfQ0/mENaImuAbxilr+EH37APlwVxpLPhh4Ny9w1rNxugf4LmtUQMw7K8AF8sRs2LRzOf
2/JKp6dhTJoXj2Z+Wu5Ae7LmleS2bCGfLhlNa7GG3eWtu9dpI5asZpXGxZXR2ZPUPXkVNoSPk0ldlh98ujpasS+u06siseK1
4nQjT1gve00ryNOkwRZpCprW6Bom6oMO4ERNS0QN+6gDB4uUFK/1lLz6hoDPkrLitfqGZomuAaf/bBSv1Rj2E4rBFldWhRKD
MbTNxhO2l2ZVLTEYw7AYQyBKl1XJxOAb9oN7AtvDmheP89Ch0bXsNS8elkXq0T4g1RA0rQnGTWENi7liOWpaaQimT1syEKTL
SdNKQ1yj1cNBLkLOmlcag7wJNKrWGMVrPSlzMxux7jkZALKK2Oob+jYPI9bR1pBitjqHsZ2VM5bxssZpamGMh/Y0k0FaxJqa
cA8JjUVZ4zU1ZRMZbVFQ1HTokMHUhYmK2uohmmVvNkBOxiRFTYcPHeYDWSOdjqgCiH2hAV1kpdcRhzNznOximdhA1spKvyOq
dErqzjjQRZY0Mx7nI/f8IJKvdJoaD/54oRYwE2sta2qLoxjaUqt7GQOZNOs1tSQ3aQOmGm3Q1NIYmMLno42a2mJn3eItAiEc
a5Oitp6daQkgO6RFWVFbgex6yAQ6m4yiJoKJbcFBxxZLVlMLY3jK40CkqYUh7lbvJ2NhybK8NbUg15pDU/GsqK0RxR7Jqdu+
AdYaeUWNhHs1RYARoKCorX6jXfxGIEZVppyglgYD2R0si3a2dEXSYCEHzxGZkNIXSYOFdGs4EFhrzmhqPE7IfvMBcI6ss5oa
D6t/Cpga5CoGaWosQwRcqQFLxDlNTVzA8fClBceaWpLns4j2kdfUxHm6B+GQfc0FRW11IfsBFu7sqKitF3Haxfx+ZARmtkuK
mjxST3FupEVZUxOH6oQGGC0bTU1ZyDqzPQJkNbVw7Gch2xGTokZGrjUC7RE7RY3GaytTwADxRZkVNXkfB74xZ1l6I1kdrT16
5rMsvZE8WEiebkJAMRXL0hvJg4Vcd1qLdHbS1Pj4LgQh8yhrajxc8igTMoLG3xtNjYdYvutAyKh5q6kt0cb1hA21iDS1NNjs
1O+ZIavfO00tzbGDvMS9kc72rKitZ+Pc8hQOtJDeK2ryqmI/iwBLzQfFbDWQti01A24iPmpmiwuZm+1vIXQAJ2liYbzu0Vca
1ENZMwtyVzPgPhuMYrYesc3qZgE4VjETOelkuksLTKJAihmJAF2PZCGnvqCEH5QD2W0IstCCcEUGID+Zx4heDBWuyAA05wPR
4EEImhrLc2gAzw8hamrCPPZNDRq1pKnxOGo9NQ0BZU0tyUAdg4e1aDS15WRMiysCAVlNLc3U3EoN2NQiKWqrA2nX3RFY/NEp
ausRm5dNDbhcZSMrato+tqgo0iKvqQXp9oN3MsuZRVMbU91TzAfxRGPU1MKYv4BvZZbdRlHTMcgEmuyYFTUaI3Xz1TEAKBlF
bTW1bsnnAlYkSV0FO/iPcUmlIcGDJNUVrLzRXawIg9tskhoLViZnXF+zyFktsabGMuQDXoyzyWtqoyOa+iVYYO2noJnxeNUq
wp5IippZGqsLpvwngJM0sSTzGAxusylrZqMbSn13RMo4slHM1oQ1L8ygOgWrqAnvsVHDMiuZFLUVyC5xYyS4mp2mFoaYKPXM
ClLCk1lTG/3HadSQjT97TS2M94X7lT2EWVDMVuto12A/0qComK3W0TVmBOaMclLMZAAyNicLuFRks/RExnoXv8SfkfoSIz0R
Uqlrgw4aGemJjBUvtIRpkMoHQ5oaz0vNNac/Q6NPxmlmPJRwxX6+QkpnDGtmLKPGhAV7yXjNLB0XGwC+OpmgqanbPAZLY5CJ
mlqSl+XAsCGZpKjJFPZ0oQfAyYrZekrvaV4w/kTWKGayENCjs8haTSzIhCF4HZUsaWYigW3AShOyThMLg92f7nsTwowVM+k7
RrQclaxXzFbr6NfME4ATFDNdC+jAGj4r3RCnjCPs8pGVfshYANOjWBntIumHOHW47l4xQo2MpsayLtVDsR4iq5mNyZkpgwE4
s0SkmYna6NT62gMWhJxmlkZmU1QVWPnEmlo6jqoapK+9pqZKpBns66CYrSfivJgQIBxGFBUzmZshtBKGKClmIjfjpoQqMmhZ
UwvjSpv2amClOaOpCevIPe0EtMhZTU3FHtFCV0eKmszNTPMR2PWdU9TEiXgKq0J9xIoameNKH+DSLDnph7CSjJjuFAMT0kk/
hAf7aJdkIVKcLN0QHnzHNUCH2FmXNLPFPIbFw0LMmsuaGY9ucQALOomNZsZ6OmL1yWw1szRWnVnYvWbSzMTZuu8gHgFymlqS
bjFjKR5iVtRkoWCEa+7ZK2qrfUztNjh4I484KGqr87hWTXsEKGpqQRqRAB7TOGlqYShZnUOPAE7WzJR5BO+bkDeKmb7cE7E7
OeStYkbm2OM3ADVPihqJSpKelwMO6eSlJzKWyIRV7gXpI+mJ+MHOmrbUwNJD8tITGYtk7GQfwRYFTU1VyRgsK08+amo8RlUd
et7zSTMTZTLtHYnzhEyjrJkJ9zH0a5SAnoTRxNJgQ2bHGAGymlmSNsSCW1EgxWw1j2YqCoByRRScoiYrB3ulHnJyDKyore5j
vR1uQe0gCl4zEyI73YYg0YcQNLMwuvyuR8OAaR2iZqZkdsCLXRSSoqZDj2jwKWRFjUZdrsSw9xiNokaq7jiBEztKTySogupJ
MgOYj1F6IkFlZvplXA9YtSg9kbH8xq1WDWkRa2ridngAEyoUvWbG46D1OgxkpcWgmbGsS2PsKjbFqJmlcdAIvY5FMWlqonam
n/gQaxSzppZkUp7BQUtGUVuP1+uehqzZZBW11T5WalMyFdiLEilqonpmEnpjBMhpamGkBlcrUGJNLYz3DeCYevKaWRhvY00+
P9KgoJjRqKYwF1wCSy1FxUymZjozRLcpKWYkNGr6xUckAJGkIxKVEtm0FwFdlKUnEpX32GcRBCQ9kTgEH93iQCB9nUlTU95j
AKdjdpoaj5IuHQjp7MyaGkuREAfujtlrauk4JILYxxw0tSQDWRaMPuaoqSUZWHWg6c9JUVv9R/8l97EoZ0VNVhd60O1zxihm
q/volvuKHtAAM1YzC9LnT2CDSBMLI7HQDyGAXpZxmlkYp2NCL5o5w4rZ6j6GNeWIUPOKmqi4ru+XYdcNnAmKGhkp72Ixl8YZ
6YmkwT7OwghQNYcz0hNJg9fXjQjoqzsjPZGkqq4nIwJI30oxVDuWznCL0YErzUk11AFoXmkWVNuTcqgDkJsK5WFqTlNLc1Le
r3qoCDXW1JI0/Q6LiDmpiGqTUmyElVWdlES1afAf4yo4BOBExWyNPq6bGiFdlBQz6T7CMWMnRVFtUtI8/ZIIIpIoVVEHoNnJ
YixK46Qs6gA0y2xiCQMnZVFtGuxj+pIrdE7qog5AYYo+Jqy6xElh1AGIR5koAEc6Innw+miJYwHbvpPKqDar0/V0iQppkXRE
8mAewxIyRiy/1Ea1eUjOmDXthABlTY3HMyij+X0n1VEHIHEPF9iLpDyqzcp9hOOqTuqjDkBzuMeA+7UUSB2A5nAPOo+kQqrN
Kvzo0fOMkxqpA1Ca7rwzqDAvVVIHIDcd1TykTuCkTKrN0u2bJaMQZkkzC+Og+S5zgLQoa2ZB6tqA+q9SKNVm5T52I4Jssqwe
hVO6tu1FYIgZq6fh1N2ehKqyOamVSkZJU3i0+M5JsdQBqFPrV9+A9SHVUsko7xH31aReKpnBe1xvvTpk9KNmJnLXUwU30qKk
qbGkBh5BnZRMJaO8x0m8DlBZNprZqMWYpqIAYPC91czSsZoekHlwUjWVxrKZ4SoFIiDtFDWpcsuoZJOTuqkDEC9CeGXJAmZW
CqfSWO1CX3AP10nh1AFn9vhBlWQnlVMHoFkdC8wVOimdSsdVM2C5rJPaqWRUXSGjVyCcFE8dgOayawtaNameSlaVXUdUgc5J
+VRS78FMKTUgGO6kfipZKfA4lc0Aoy/1U8ke5WbAlyKcFFClsWqmZzBQrfagiY03H+enEJAGRc1MvQvjwQYlTSyNd0T6hV5o
7LNmNt4Kd73cAYjyO6mhSvZIAhyUSXFSRHUASpMNyVjO0UkZ1QFovrQEvpbkpI4qWXUvPHZHHWkRa2rKPDI4alJJleyRKgVh
Do1UUqXxiRj+kusvTkqpkpWqkVMUC4mGSi3VAWguckV3aymmSnSUusaKE51UUyX1SswU5AceNHBST3UAmjdrB05HKahKY7GL
X9+JAUZfKqoSHV3sQd98kJKqREq1Zyq/QYC8piZeR2A4qC5VVYlUambyjIBpJGVV6bhqJmKxHimrSnSUuW4rFgHKitkaemxP
nrZSacQYSWHVAciNKsrANJLKqkQqc009yo88HUKamroYTmCkT2qrEsl3EmadPaSPWFEj84jSNDBqUl2VSN4RShFes1JflcbC
mbUGA8kWSYFVGh+O6dUl4K0lJxVWyamL4Z0a4mFLiVVyMnc9VSoAZ1mWGqvkpK73dMUccB9ZaqySk/IW81WKBACRpsbHCV5C
qDlNLY0TssewkWdxpMbqABSmbQ3UmGepsUqPV84gfR0UM2kgGX17gaXEKrmj1DX4vCBLiVVyUnB8SoMiY5Y1szDKG/SkPDId
pcTqANQLuSJqjVhKrA5APB1BwZpStuqFe/XEIKMbP1v1zr0SNfPom2wsJVbp+CkZg01HqbBKrA7Xk2400kXSExnfklkvhhPy
kJX0RFipPk4Kq0gXJU1NBB+7kwW8IsRSYZXGwpnc5iP6WpNUWB2AeKCGrH2psErjgzJ5ycwhVk0qrNL4EoxZzzMAjtPMxM3H
hFaVshRYpeMnZcAbYiwFVkm/KTNVliF9HRQ1HXyM4EtvUmCVxsIZO7nYxX9E+ihpaiI3E1puJiN9lDW1II2IBV82lgKr9NjD
MpCzzlJglVhdfZycLMD0S4HVAag9pjhFDpDNSAqs0lg5477khWOWAqsD0HrVDFKzZSmwSv5IlQI78bPUV6WxcMav9wORQYua
mbgaPimAAEtN6quSqpxx0wM8yOhnTU09uJUxAXKW+qo0ls7kttRA/UCW+qrk5dXH6aiGuNhSX5W8eo2V++gDfST1VQegfp7p
2XTkLUOprzoAzXdxMxZWZ6mvSl5VXve3OQ2AEzSzMApQd2UCZHeU8qoDEE8WG8vxsVRXJX/0DqGH3stiKa464PQ9ra9YZE+T
4qrkj6KPqGMsxVUpqNN17yHkLCvFVSlISdy2XUMRWpbiqjS+NxOXinIgwctSXJV04QxcFcJSXJXGypk5kgUlwliqq1KQ7zPg
16dZqqtSkFHDyckCkiEs5VUFEDUbAr6YxlJelYK6+di3EMQTkfKqFNTN8F7riFg1qa86AIXpLkUAR03qq9JYOcNLwYtDgJym
FmRNOWHmUcqrUlDmMbcGIW/hek1MSJr1iAjiqkt1VQoq+jh1NWBnpbrqANSDPdOdDMAaSXXVAWh+L4JBj0aqq1JUko+49yjV
VWmsnOnTEbz1zFJdlcZnZ7rPj8b6pLoqRaWJOwkZI28hO01tTM/kHoJANiOprkpRPdc6PTCPdLbX1NSjCgTOI6muSlG5jxG9
ccBSXZV05UxEo9gs1VUpHrmP8KhlRU2mryP6gAlLdVUaS2d4kcpATL+UV6WoLvdMAVGgj6S8KkUlTJH7qAETUsqrCqBVDh3Z
HqW8Ko21M7xcFIGAvKImb/d0f80j1IKiplUfwRQ/S31VSup87eETvxRYpaRqCyOq0M5SYJWSer41wBNSCqxSGlQf7ZJ6Qg6P
UmCVkvIgpzMfQE0KrNJYO7PqiSDzSAqs0vjsjFlPNMDwS4FVSkq6Z4pkIp3tNTX17EwCg7RSYZWSuhzO6B0olgqrNBbPzLEs
sI+Sonb8ciHmH0mFVUpHFhIsvfdSYZXGZ2fMMmrA1WcvFVYpyQz2fA0GeDleKqxSUhYyoi/FeSmxSrp6hrvYWgaAWFFbgfwa
o0eApDein51hVAnES4nVAShM5TOEybd4KbFK+ZHqQsjR8lJilcbymbXmySHUsqYmtB+nmCgAJDVWKcunC2fFpQgAWU0t6Uv9
5TAKDL8UWaV8FIIE1f+8VFmlsXxmjfYDR2MvVVYpqwvirschAByvmInnaxKstualyuoANM9HMKvqpcoqjc/ODJLoSF8nTS1I
rT1GBy1rakGWhIL3BL1UWaWsDGQ/iQI3KryUWaWs1HsC6mZ7KbNK48MzvOq2OQBIOCPOHL3LBUrreymz6ow8G0+JDOC45qXM
qgDyi6oEEDv0UmfVGWUgGX1d3EudVWdUfWHu2SeEWtLUWFaqgQ8Oeqmz6saHZ9x65wwAkjqrzsgnWVPPqiMTUuqsurGCZq4K
RcL0XsqsOqOedg2ojqCXMqsDUF9qCVWQ9VJm1Zmj8mvQp/VSZtUZ9S5Xvy/ACLWgqY0vNOTpSR1gw5Y6q258d2ZVsoc6Oylq
ssCQwWSPlzKrAse06QgWKnqpszoAjc/EtdF//ubi2duLzdtn37y82Py8Pezu3h1294f7zddfbcqf+s/v9h8285/ygxd/uXiz
+e7Ni1fP3vyw+dvFD2ftg+9v91flu8tH5w++/vbt5vXfX77cvLn488Wbi9fPL76fPnv/9fKd35+tP/ahtGHCeHvxz7cLQP/I
/f5qf7l99/nzdf9Iaf3L9SPlL54+nT6z2d9sDp920491ameb8sWGc3l78+Hh8rD/1/7wy7uH+0dxxs+cba73l3e39/vd9e7m
/j8uO8rnT7f3nz8V5KlBj6Esn9nc7e73Hx62V2srPn/aiD/i+1/pYR5GZ335QrwIZ5+UM11dRzG1rHC7pmweyX09BiWfKCxQ
zR6Tq+cyqtrF1j6yyT+GJN8oLEgtEEe25YWr/n1pU3QIknykMFWkYo5MsLVNba+3j+yFjyHJVwoLUrtXRL45063UyR77nY8B
yWcKU+0mWrqp+XqlwwOCJOutfSVXkTh3cq2bAtRNsuC6INl2rTC0THwZu1yhEtQoWXJdoOp+QblFnMhNPc7Q1JReI5VWNfXa
6GInSG0eQDNKKsA0rPaOgktxmQnmkWKFR7Gk79ixyt853+QEmlNTOh5rl4xl5dpdZePNpgUNWqrfPpLpexRKRrNy664yGj62
pdyuQppHtA4exZIBrdwolr9jdr272oSIdWd7/u2rVy/e/vF/AcpQ4NLIvwAA
"""

if os.path.exists("boilerhouse.db"):
    os.remove("boilerhouse.db")

_sql = gzip.decompress(base64.b64decode("".join(_DATA.split()))).decode()
_con = sqlite3.connect("boilerhouse.db")
_con.executescript(_sql)
_con.commit()
_con.close()

print("Database ready:", round(os.path.getsize("boilerhouse.db") / 1024), "KB")
print("Now run the next cell.")

In [ ]:
import sqlite3, base64, gzip, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

con = sqlite3.connect("boilerhouse.db")

_BANK = json.loads(gzip.decompress(base64.b64decode("""
H4sIAKDolmoC/81Z/W7bNhB/lUOAwXGhqv6Igy3dBqSJl7ptnCFO2w0IoMoWbauRRVek42ZFgT3EnnBPsjtSkimJ/kqxYfrD
sGiSd/e77/OXg6bbPDiBLwfTMJb45eBmykD6w4hBKODDkIcRS8SH5yB4ImEZyilcXZ93r+HF75D4MownnpxP4bw7OHMPHDgQ
nyK6ZdB90z27AYARD5gDsT/Dz5l/xxJvhiuRYxx2cNNsFgoR8pgFt/Ev11eXQE9K/DZeR/HgK1Jsuq2CAO9fdq+7IKbhWEKv
D4e1FzWndlarA/z951/AEwBzx0+Av6NIq9ezmgsIQrySc7yIIo9EsErIZnMv5s5ql6PvMgThc4accxJF08anyCCRrRuC5pel
IrYLIp5dve3fHD6pI3CLWApI+FI8B7163hvc9PrImUbPC4N8WxCOxyxhsYR7P1ow4cILjvqccAhjkKh3gQRBC2YRNadafU4H
EHsJ8wPUjnBu43R5PUf6RK7gFKnshlToo4LQOTZC+qM7TxLuI2UGDjEfw5veZe8GOlYl5aRxr3AKVxh6yugbiqgQu401HXw6
KZ8di/3lBNGkWnDaP0e6yOHrLtRajdbx08bR00bzu5o2ShF+zmk7gF4A7J4lDzDmiwSm+CEsQj2pwJZbV4E4aYPo2xlIRTgu
xoAlR5uJg1CiSwr4yEN0TBg+0D0nRUh+huYPDXU/b3nzkYQfoe02Lexu0ICTnl0vUJVkLpVJVgnTKgW003cXh0Iyf0Zho46m
LsKAwTWa5vmh67oONOsu9Dlc4NKvqPITpYB4MRuyBDWQgB9F0DlqKC+zCKZvKlKhS8nEZ8yPvXzZbuetUvzK3Yxok1ciXe3B
DvhxkPNpaHkRByyJmS+nW4w/90wosL0Ct8w5/TIyj20IAsUoYHWrKu8pBO1KDhJLPEI56PL0tyKL8BQue/3SGrnRMgklw6sx
po0X8Ugbrz/xMbqlWic4leD74lQhmOE04lHAhLScKLOdnZhyKXc7YRM0u0XBsx/GxXB64QvAtASSx7HK9YO3l4fjBYu8ycc6
PFOvqeVqdCWXfkRJ5B6RVNFAr2B25xItE3yMWf6EAR+rPM3FviCbJJWcVftSJDOPst+QybDxhnTT5htKKDjQVtBPPnoInKeA
MzQQ+GH04EXcrgMjpZjnV2VMq5RGyAdUUYResORJQCp6efqu179woSdhHEYSUydMEr6YY/LXgTJbXROqCnlwjwCwo53dxppB
Arp8LUXtTiOVtJht8osQOy/wJUvzeQ4Yea1SG6CrjVhMDKhQqDNx2yrp6rI1xkVOhPEiM6ZtmszuM2sD43i5NmgrUdulXPTq
Cgu+tOqBIVzhi2sm62T1ZlefqwvMxKUcmrhb6hhIbmNF0qimt5A1i203vUct55I1G6lorWrMZlQ1qHqBzDUvKcXUnzPw9co4
TIQ8AcVXXhgDJ664my1ovgLzfQ0gusHgrlF+c1cV1w6eX1lB4G5SNAQWpKCKFSiuDLTyQ/uIYoCsWbQDXEyKrwhVBV7qHysH
VCCoIP3AFzokUHxeTn2pVsSULzfCt4oFRZvaNRo8ytBK/BugmNRWEbJdymADMi00KsxsDgwXUttdRr1YKikKLgyoiTWCiODK
JDG+CkkWy7EP4vMdoKJoErilRJGb2O6pwmp3FuCC3YCzp5Z2KbVkes5r4md4k9FaP0ETbLhwkdlRal+EaGFfip5egRGWAEOm
bC3ehKCRcs3b7LWpyaYtoWdZvVhob7/LKnJm7NhPeHzs6Z//JWt3CvQNFeqfU70dV7PHPnGmHCjSwGjVThY0rRXRnqa+a8Ng
aR901t3mKfsF27L8BtrpioL7qJSsz04HXaqr+pjoo3Dke/P5jMqYBtzQYm3py9G0Bt03uK3G7/Ab9qNUVUhfLoSumdlnKo3F
kiwkZlg2Uz0XMfi04NgEbB1VYJ+Qpq8VCwVwv4lFA2fcSMpDeubww6BfbCaOLMk/HxiYzZZiEANriT2MxIvZjKJGU0fqhsiG
VGnHS18we81IgN37CDLNLZg0NRoNgqKubY5j4+JF4SyUjzNc036x3plHbC20a5qzo1KuL8lRmn5837GI4sJrxuYKPoreDWV/
+A38iOYZk/CeCdSMZBNs+qiJo3Grwv4Pluzfq2kaT+ARnGIIybAtRFzSAp7Yp6k9KpUE6WS3yMVhKk+1J1GEMiJpl4v4DRPc
xCQki1joikvhlDC5SGJhjIe2+W9uQso0/CGK6KmmIW2VraKuJsSPE2MDWB1735XX6ezzPGFqDq/LRhaxkWQBjfySsQxn7LD2
3fsaDfDqVtGr2wresWTs7j8YQW1GtTQbzUGo8m4EQuI8xfC4WpRnJYDkMEGzkSoYBqycfVWliReZMOuaKi/f5ZJKKlVjpCP5
7PQueK/aHdvgo4x+tRz7P5W260VzqmVvVlLlaupY/tRScRiGER/drW9OOX4kpIYTqM6tDXvNMwuy0zKsdZVxFCJG7LfMbV3o
+qOpbltUomOqA7GOAN73bl4COYggYodaa9881qFLLM5Cb7bxTh2thQxjJxb2nqQVmDFnMJvY0VrdGZKteiuxUUjea/m4jStt
onQL4QvGbqFAhqVbIVz+3xOGhgNp1UtyH1lwn6F1GqG0BGPaPt5hu0ZxSduX67ev/p10LY3m138Ay4+izkweAAA=
""".replace("\n", ""))).decode())


def q(sql):
    """Run a SQL query and hand back the answer as a table."""
    try:
        return pd.read_sql_query(sql, con)
    except Exception as e:
        print("SQL did not run:", str(e).strip().splitlines()[-1].split(": ", 1)[-1])
        if "___" in sql:
            print("(there is still a ___ blank in that query)")
        return pd.DataFrame()


def _rows(df):
    """Turn a table into something comparable: row order and column order do not matter."""
    out = []
    for _, row in df.iterrows():
        cells = []
        for v in row:
            try:
                cells.append(("n", float(v)))
            except (TypeError, ValueError):
                cells.append(("s", str(v).strip().lower()))
        out.append(sorted(cells, key=lambda c: (c[0], c[1])))
    return sorted(out, key=repr)


def _same(got, want):
    if len(got) != len(want):
        return False
    for a, b in zip(got, want):
        if len(a) != len(b):
            return False
        for (ta, va), (tb, vb) in zip(a, b):
            if ta != tb:
                return False
            if ta == "n" and abs(va - vb) > max(0.06, 0.005 * abs(vb)):
                return False
            if ta == "s" and va != vb:
                return False
    return True


def check(tag, result):
    """Show your answer, then say whether it is right. Column names are ignored."""
    display(result)
    want = pd.read_sql_query(_BANK[tag]["sql"], con)
    if result is None or len(result) == 0:
        print(f"[{tag}] nothing to check yet — the answer has "
              f"{len(want)} row(s) and {want.shape[1]} column(s).")
    elif _same(_rows(result), _rows(want)):
        print(f"[{tag}] correct — {len(want)} row(s), {want.shape[1]} column(s).  OK")
    else:
        print(f"[{tag}] not there yet. The answer has {len(want)} row(s) x {want.shape[1]} "
              f"column(s); you have {len(result)} x {result.shape[1]}.")
        print(f"      Try  hint('{tag}')  — or  solution('{tag}')  once you have really tried.")


def hint(tag):
    print(_BANK[tag]["hint"])


def solution(tag):
    print(_BANK[tag]["sql"])


# A first look. Change 'boilers' to any other table name and run it again.
q("SELECT * FROM boilers")

That is the whole mechanism: you write SQL between the quotes, and `q(...)` hands you back a
table. Three tools go with it:

* `check("1.1", q("""..."""))` — runs your query, shows the table, and tells you whether it is right.
  It ignores column names, column order and row order; it cares about the values.
* `hint("1.1")` — one line of help.
* `solution("1.1")` — one correct answer. Use it *after* you have tried, not instead.

**Two rules that catch everybody:**

* Text goes in single quotes — `code = 'B-2103'` is right, `code = "B-2103"` is not.
* Numbers do not — `boiler_id = 2`, `rating_tph > 30`.

In [ ]:
# The other small table. Six people, three shifts.
q("SELECT * FROM operators")

In [ ]:
# And the first few rows of the three bigger tables.
display(q("SELECT * FROM readings    LIMIT 5"))
display(q("SELECT * FROM daily_logs  LIMIT 5"))
display(q("SELECT * FROM water_tests LIMIT 5"))

> **Look carefully at `readings`.** It never names a boiler — it says `boiler_id = 2`. That
> number is an internal id, and it is **not** the tag painted on the boiler: `boiler_id = 2` is
> tag `B-2103`, not `B-2102`. The only way to know which machine a row belongs to is to look the
> id up in `boilers`.
>
> Storing the boiler's tag, name, maker and rating *once*, in one place, and pointing at them
> from everywhere else, is the whole idea of a database. Guessing from the id is exactly the
> habit this notebook is trying to break. We put the two back together in Exercise 3.

---
# Exercise 1 · Reading the data

`SELECT` · `WHERE` · `AND` / `OR` · `ORDER BY` · `LIMIT` · `COUNT` · `DISTINCT`

**Worked example first.** Pick some columns, keep some rows, sort the result:

In [ ]:
q("""
SELECT   code, name, rating_tph
FROM     boilers
WHERE    rating_tph > 30
ORDER BY code
""")

### 1.1 The asset register: `code`, `name`, `maker_model`, `rating_tph`, `commissioned` — biggest first.

Then answer out loud: what does one row represent? And **which two of the three are sister
units** — identical machines that can fairly be compared with each other?

In [ ]:
# YOUR TURN 1.1
check("1.1", q("""
SELECT   code, name, maker_model, rating_tph, commissioned
FROM     ___
ORDER BY ___ DESC
"""))

### 1.2 The operators on shift **B or C** — `emp_no`, `full_name` and `shift`, in alphabetical order.

Either `shift IN ('B', 'C')` or `shift = 'B' OR shift = 'C'`. Note that `shift = 'B' OR 'C'`
is *not* SQL, even though it is how you would say it out loud.

In [ ]:
# YOUR TURN 1.2
check("1.2", q("""
SELECT   emp_no, full_name, shift
FROM     operators
WHERE    ___
ORDER BY ___
"""))

### 1.3 In one row: how many readings are there, and how many different boilers do they cover?

Two aggregates side by side in the same `SELECT`. *Predict the first number before you run it:*
3 boilers × 6 readings a day × 30 days.

In [ ]:
# YOUR TURN 1.3
check("1.3", q("""
SELECT COUNT(*) AS n_readings,
       ___      AS n_boilers
FROM   readings
"""))

### 1.4 The five hottest stack temperatures — `boiler_id`, `ts`, `stack_temp_c`, hottest first.

**Look at which boiler they belong to.** Remember it.

In [ ]:
# YOUR TURN 1.4
check("1.4", q("""
SELECT   boiler_id, ts, stack_temp_c
FROM     readings
ORDER BY ___
LIMIT    ___
"""))

### 1.5 Every reading for `boiler_id = 2` taken on 1 April 2026 — all columns.

The `ts` column holds a date *and* a time, so match the start of the text:
`ts LIKE '2026-04-01%'`. Join the two conditions with `AND`.

In [ ]:
# YOUR TURN 1.5
check("1.5", q("""
SELECT *
FROM   readings
WHERE  boiler_id = 2
  AND  ___
"""))

### 1.6 A needle in the haystack

Out of 540 readings, find the ones that are **hotter than 190 °C *and* below 3.1 % oxygen**.
Show `boiler_id`, `ts`, `stack_temp_c`, `o2_pct`.

Why that pair of conditions? Hot flue gas *with plenty* of oxygen usually just means too much
combustion air — an easy fix. Hot flue gas with **normal** oxygen means the heat never made it
into the water at all. That is a different, more expensive problem.

In [ ]:
# YOUR TURN 1.6
check("1.6", q("""
SELECT boiler_id, ts, stack_temp_c, o2_pct
FROM   readings
WHERE  ___
"""))

---
# Exercise 2 · Summarising

`AVG` · `MIN` · `MAX` · `SUM` · `COUNT` · `GROUP BY` · `HAVING`

**Worked example.** One number out of 540 rows — and then one number *per boiler*:

In [ ]:
display(q("""
SELECT ROUND(AVG(steam_tph), 1) AS mean_steam_tph
FROM   readings
"""))

display(q("""
SELECT   boiler_id,
         ROUND(AVG(steam_tph), 1) AS mean_steam_tph,
         COUNT(*)                 AS n_readings
FROM     readings
GROUP BY boiler_id
"""))

`GROUP BY boiler_id` tells the database to deal the rows into piles — one pile per boiler —
and answer your question separately for each pile.

`COUNT(*)` is there as a check: 180 readings went into each average. If one boiler had 40,
you would want to know why *before* trusting the number next to it.

### 2.1 The average steam flow across every reading, to one decimal place.

In [ ]:
# YOUR TURN 2.1
check("2.1", q("""
SELECT ROUND(___, 1) AS mean_steam_tph
FROM   readings
"""))

### 2.2 The average **stack temperature** for each boiler, with the row count beside it.

**Look hard at the answer.** One of the three is not like the others. Say out loud how many
degrees hotter it runs — you will need that number later.

In [ ]:
# YOUR TURN 2.2
check("2.2", q("""
SELECT   boiler_id,
         ROUND(AVG(stack_temp_c), 1) AS mean_stack_c,
         ___                         AS n_readings
FROM     readings
GROUP BY ___
"""))

### 2.3 For each boiler: the lowest, the highest, and the **swing** between them.

The swing is `MAX(...) - MIN(...)`. A boiler whose stack temperature wanders over a wide range
is telling you something different from one that sits at a steady wrong number.

In [ ]:
# YOUR TURN 2.3
check("2.3", q("""
SELECT   boiler_id,
         ROUND(MIN(stack_temp_c), 1) AS coldest,
         ROUND(MAX(stack_temp_c), 1) AS hottest,
         ROUND(___ - ___, 1)         AS swing
FROM     readings
GROUP BY boiler_id
"""))

### 2.4 From `daily_logs`: total steam, total gas, and **gas per tonne of steam** for each boiler — worst first.

Gas per tonne is `SUM(fuel_gj) / SUM(steam_t)`.

> **Why not `AVG(fuel_gj / steam_t)`?** Because that treats a quiet 400-tonne day and a flat-out
> 700-tonne day as equally important. Totals divided by totals weights each day by how much
> steam it actually made. The two answers differ, and only one of them is the plant's real fuel bill.

In [ ]:
# YOUR TURN 2.4
check("2.4", q("""
SELECT   boiler_id,
         ROUND(SUM(steam_t), 1) AS total_steam_t,
         ROUND(SUM(fuel_gj), 1) AS total_fuel_gj,
         ROUND(___ / ___, 3)    AS gj_per_tonne
FROM     daily_logs
GROUP BY boiler_id
ORDER BY gj_per_tonne DESC
"""))

### 2.5 Which boilers have an average stack temperature above 150 °C?

`WHERE` throws away **rows before** they are grouped. To throw away **whole groups** you need
`HAVING`, which goes after `GROUP BY`. `WHERE AVG(stack_temp_c) > 150` is an error — try it if
you like, the message is worth seeing.

One word goes in the blank.

In [ ]:
# YOUR TURN 2.5
check("2.5", q("""
SELECT   boiler_id, ROUND(AVG(stack_temp_c), 1) AS mean_stack_c
FROM     readings
GROUP BY boiler_id
___      AVG(stack_temp_c) > 150
"""))

### 2.6 The three busiest days on the site

Groups do not have to be boilers. Group `daily_logs` by `log_date` instead, total the steam from
all three boilers, and show the three biggest days.

In [ ]:
# YOUR TURN 2.6
check("2.6", q("""
SELECT   log_date, ROUND(SUM(steam_t), 1) AS site_steam_t
FROM     daily_logs
GROUP BY ___
ORDER BY ___ DESC
LIMIT    3
"""))

**A picture of what you just found.** Run this cell once you have done 2.2:

In [ ]:
daily = q("""
SELECT   substr(r.ts, 1, 10) AS day,
         b.code,
         AVG(r.stack_temp_c) AS stack_c
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
GROUP BY day, b.code
ORDER BY day
""")

fig, ax = plt.subplots(figsize=(10, 3.6))
for code_, colour in [("B-2101", "#1C7293"), ("B-2102", "#0B3C49"), ("B-2103", "#E07A3F")]:
    part = daily[daily["code"] == code_]
    ax.plot(range(len(part)), part["stack_c"], lw=2, color=colour, label=code_)
ax.set_ylabel("stack temperature, °C")
ax.set_xlabel("day of April")
ax.set_title("Daily average stack temperature")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

One line sits well above the other two — you already knew that from 2.2. Look again, though:
is that line **flat**? Hold that thought until Exercise 4.

---
# Exercise 3 · Joining tables

`JOIN ... ON`

`readings` only knows `boiler_id = 2`. A **JOIN** is how you look that `2` up in the
`boilers` table and glue the two rows together.

**Worked example.** Read the `ON` line as a sentence: *where these two numbers match, that is
the same boiler.*

In [ ]:
q("""
SELECT b.name, r.ts, r.stack_temp_c
FROM   readings r
JOIN   boilers  b  ON  b.boiler_id = r.boiler_id
LIMIT  10
""")

The letters `r` and `b` are just nicknames, so you can write `r.ts` instead of
`readings.ts`. You choose them yourself.

### 3.1 The first 10 readings with the boiler's **name** instead of its number.

In [ ]:
# YOUR TURN 3.1
check("3.1", q("""
SELECT   b.name, r.ts, r.stack_temp_c
FROM     readings r
JOIN     ___ b  ON  ___
ORDER BY r.reading_id
LIMIT    10
"""))

### 3.2 The first 10 daily logs with the boiler **code**, the **operator's name** and their **shift**.

Two JOINs — one to `boilers`, one to `operators`. The second is the same line as the first,
written again with different names.

In [ ]:
# YOUR TURN 3.2
check("3.2", q("""
SELECT   b.code, o.full_name, o.shift, d.log_date, d.steam_t
FROM     daily_logs d
JOIN     boilers b  ON  b.boiler_id = d.boiler_id
JOIN     ___     o  ON  ___
ORDER BY d.log_id
LIMIT    10
"""))

### 3.3 Average stack temperature per boiler again — but showing the boiler **code**, hottest first.

In [ ]:
# YOUR TURN 3.3
check("3.3", q("""
SELECT   b.code, ROUND(AVG(r.stack_temp_c), 1) AS mean_stack_c
FROM     readings r
JOIN     boilers b ON ___
GROUP BY ___
ORDER BY mean_stack_c DESC
"""))

### 3.4 Gas per tonne of steam per boiler **code**, worst at the top.

This is 2.4 again with a join on top. Nothing is filled in for you this time.

**Which boiler is worst, and by how much?** Work out the percentage in your head before moving on.

In [ ]:
# YOUR TURN 3.4
check("3.4", q("""
SELECT   ___
FROM     daily_logs d
___
"""))

### 3.5 Is the bad boiler simply being worked harder?

A fair comparison needs **load**, not just output. For each boiler show `rating_tph`, the average
`steam_tph`, and the average as a **percentage of the rating**:
`AVG(r.steam_tph) / b.rating_tph * 100`.

This one matters. If the hot boiler were running flat out, a high stack temperature would be
normal and there would be nothing to investigate. Is it?

In [ ]:
# YOUR TURN 3.5
check("3.5", q("""
SELECT   b.code,
         b.rating_tph,
         ROUND(AVG(r.steam_tph), 1) AS mean_steam_tph,
         ROUND(___, 1)              AS pct_of_rating
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
GROUP BY b.code, b.rating_tph
"""))

### 3.6 Rule out the people

Maybe one shift simply runs the plant badly. For each **shift** show gas per tonne
(`SUM(d.fuel_gj) / SUM(d.steam_t)`) and how many logs it covers.

If the three numbers come out the same, you have eliminated a suspect. Eliminating a suspect is
a result, even though the table looks boring.

In [ ]:
# YOUR TURN 3.6
check("3.6", q("""
SELECT   o.shift,
         ROUND(SUM(d.fuel_gj) / SUM(d.steam_t), 3) AS gj_per_tonne,
         COUNT(*)                                  AS n_logs
FROM     daily_logs d
JOIN     ___ o ON ___
GROUP BY ___
"""))

---
# Exercise 4 · Sharper questions

`CASE WHEN` · sub-queries · dates

From here on the starter cells are emptier. That is deliberate — you have seen every piece you
need.

### 4.1 Flag the water samples

The plant's limit is **10 ppm silica**. Show `boiler_id`, `test_date`, `silica_ppm`, and a
column called `status` that reads `'watch'` above the limit and `'ok'` below it:

```
CASE WHEN silica_ppm > 10 THEN 'watch' ELSE 'ok' END AS status
```

`CASE` is SQL's *if*. It writes a new column that was never stored in the table.

In [ ]:
# YOUR TURN 4.1
check("4.1", q("""
SELECT   boiler_id, test_date, silica_ppm,
         ___ AS status
FROM     water_tests
ORDER BY test_date, boiler_id
"""))

### 4.2 Count the flags

One row per boiler: how many of its 5 samples were over the limit, and how many samples there
were. A `CASE` inside a `SUM` turns a condition into a count:

```
SUM(CASE WHEN silica_ppm > 10 THEN 1 ELSE 0 END)
```

**Then judge the evidence.** Does the water data on its own convict anybody? Five samples per
boiler is not much to go on — say why out loud before you continue.

In [ ]:
# YOUR TURN 4.2
check("4.2", q("""
SELECT   boiler_id,
         SUM(CASE WHEN ___ THEN 1 ELSE 0 END) AS n_over_limit,
         COUNT(*)                             AS n_samples
FROM     water_tests
GROUP BY boiler_id
"""))

### 4.3 What fraction of the time is each boiler above 185 °C?

An average hides how often. Give the **percentage** of each boiler's readings above 185 °C.

> Keep the `100.0`, not `100`. In SQL `3 / 4` is `0` — whole numbers divide into whole numbers.
> `100.0` drags the sum into decimals. This silently returns zeros for a lot of people.

In [ ]:
# YOUR TURN 4.3
check("4.3", q("""
SELECT   boiler_id,
         ROUND(100.0 * ___ / COUNT(*), 1) AS pct_over_185
FROM     readings
GROUP BY boiler_id
"""))

### 4.4 Compare rows against a number you have not worked out yet

How many readings from each boiler are hotter than the **site-wide average** stack temperature?

You cannot type the average in — it belongs inside the query:

```
WHERE stack_temp_c > (SELECT AVG(stack_temp_c) FROM readings)
```

The bracketed query runs first and hands back a single number.

**The result is lopsided.** Ask yourself what that means for a plant that alarms on one
site-wide limit: which boiler would never trip, and which would never stop?

In [ ]:
# YOUR TURN 4.4
check("4.4", q("""
SELECT   boiler_id, COUNT(*) AS n_above_site_average
FROM     readings
WHERE    ___
GROUP BY boiler_id
"""))

### 4.5 Is it getting worse?

A month-long average hides a trend. Group the readings for `boiler_id = 2` **by week** and watch
the number move: week, average stack temperature, and how many readings are in each week.

```
strftime('%W', ts)     the week number of a date
substr(ts, 1, 10)      just the date part, 'YYYY-MM-DD'
```

Whatever you put in `SELECT` you must repeat in `GROUP BY`.

Two of the weeks are short — April does not begin on a Monday, which is why you were asked for
the count. Does that weaken the story or not?

**Read the numbers from top to bottom before going on.** Is this a boiler that is *bad*, or a
boiler that is *going bad*? Those two need different work orders.

In [ ]:
# YOUR TURN 4.5
check("4.5", q("""
SELECT   strftime('%W', ts)          AS week,
         ROUND(AVG(stack_temp_c), 1) AS mean_stack_c,
         COUNT(*)                    AS n
FROM     readings
WHERE    boiler_id = 2
GROUP BY ___
ORDER BY week
"""))

### 4.6 Does the gas meter agree?

Now the same question of `daily_logs`, for **all three** boilers at once: week, boiler `code`,
and gas per tonne. Two things in the `GROUP BY`.

Then look down each boiler's column. One should be climbing while the other two sit still.

This is the heart of the day. The thermocouple and the gas meter are different instruments, on
different tables, filled in by different people — and they are telling you the same story.

In [ ]:
# YOUR TURN 4.6
check("4.6", q("""
SELECT   strftime('%W', d.log_date) AS week,
         b.code,
         ROUND(___, 3)              AS gj_per_tonne
FROM     daily_logs d
JOIN     ___
GROUP BY ___, ___
ORDER BY b.code, week
"""))

**The trend, drawn.** Run this once 4.5 and 4.6 are working:

In [ ]:
wk_temp = q("""
SELECT   strftime('%W', r.ts) AS week, b.code, AVG(r.stack_temp_c) AS stack_c
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
GROUP BY week, b.code
ORDER BY week
""")

wk_fuel = q("""
SELECT   strftime('%W', d.log_date) AS week, b.code,
         SUM(d.fuel_gj) / SUM(d.steam_t) AS gj_per_tonne
FROM     daily_logs d
JOIN     boilers b ON b.boiler_id = d.boiler_id
GROUP BY week, b.code
ORDER BY week
""")

colours = {"B-2101": "#1C7293", "B-2102": "#0B3C49", "B-2103": "#E07A3F"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

for code_, colour in colours.items():
    part = wk_temp[wk_temp["code"] == code_]
    ax1.plot(part["week"], part["stack_c"], "o-", color=colour, lw=2, label=code_)
    part = wk_fuel[wk_fuel["code"] == code_]
    ax2.plot(part["week"], part["gj_per_tonne"], "o-", color=colour, lw=2, label=code_)

ax1.set_title("Stack temperature by week")
ax1.set_ylabel("°C")
ax2.set_title("Gas per tonne of steam by week")
ax2.set_ylabel("GJ / t")
for ax in (ax1, ax2):
    ax.set_xlabel("week number")
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()

---
# Exercise 5 · One table that makes the case

`WITH` — a sub-query with a name

Your supervisor will not read six tables. They will read **one**, with one row per boiler:

| code | mean_stack_c | gj_per_tonne | mean_silica_ppm |
|---|---|---|---|

The three numbers live in three different tables, summarised three different ways. `WITH` lets
you work each one out separately, give it a name, and then join the names together as if they
were tables:

```
WITH temps AS ( ...one row per boiler... ),
     fuel  AS ( ...one row per boiler... )
SELECT ...
FROM   boilers b
JOIN   temps t ON t.boiler_id = b.boiler_id
JOIN   fuel  f ON f.boiler_id = b.boiler_id
```

Two of the three blocks are written for you. Write the third, join them up, and sort so the
worst boiler is on top.

In [ ]:
# YOUR TURN 5.1
check("5.1", q("""
WITH temps AS (
    SELECT   boiler_id, ROUND(AVG(stack_temp_c), 1) AS mean_stack_c
    FROM     readings
    GROUP BY boiler_id
),
fuel AS (
    SELECT   boiler_id, ROUND(SUM(fuel_gj) / SUM(steam_t), 3) AS gj_per_tonne
    FROM     daily_logs
    GROUP BY boiler_id
),
water AS (
    ___
)
SELECT   b.code, t.mean_stack_c, f.gj_per_tonne, w.mean_silica_ppm
FROM     boilers b
JOIN     temps t ON t.boiler_id = b.boiler_id
JOIN     fuel  f ON ___
JOIN     water w ON ___
ORDER BY ___ DESC
"""))

### 5.2 No query for this one — write the finding

You now have everything. In the cell below, write **four sentences**:

1. Which boiler, and what you think is physically wrong with it.
2. The two independent numbers that support you, and where each came from.
3. How fast it is getting worse — use your weekly numbers from 4.5 and 4.6.
4. What would change your mind: what result would tell you that you are wrong?

*Hint for (1): if the heat is not going into the steam, where is it going, and what stops heat
crossing a tube wall?*

The fourth sentence is the one engineers skip and the one that matters. An explanation that
nothing could disprove is not an explanation.

In [ ]:
answer = """

1. ...
2. ...
3. ...
4. ...

"""
print(answer)

---
## Check yourself

Run this once you have finished. It works out the numbers and compares them with the rule of
thumb engineers use: **a boiler loses roughly 1 % of its efficiency for every 20 °C of extra
flue gas temperature.**

In [ ]:
both = q("""
WITH temps AS (
    SELECT   boiler_id, AVG(stack_temp_c) AS mean_stack_c
    FROM     readings
    GROUP BY boiler_id
),
fuel AS (
    SELECT   boiler_id, SUM(fuel_gj) / SUM(steam_t) AS gj_per_tonne
    FROM     daily_logs
    GROUP BY boiler_id
)
SELECT   b.code, ROUND(t.mean_stack_c, 1) AS mean_stack_c,
         ROUND(f.gj_per_tonne, 3) AS gj_per_tonne
FROM     boilers b
JOIN     temps t ON t.boiler_id = b.boiler_id
JOIN     fuel  f ON f.boiler_id = b.boiler_id
ORDER BY gj_per_tonne DESC
""")
display(both)

worst = both.loc[both["gj_per_tonne"].idxmax(), "code"]
rest = both[both["code"] != worst]
d_temp = both.loc[both["code"] == worst, "mean_stack_c"].iloc[0] - rest["mean_stack_c"].mean()
d_fuel = both.loc[both["code"] == worst, "gj_per_tonne"].iloc[0] / rest["gj_per_tonne"].mean() - 1

print(f"\n1) THE LEVEL")
print(f"   {worst} runs {d_temp:.0f} °C hotter at the stack than the other two.")
print(f"   {worst} burns {d_fuel*100:.1f} % more gas per tonne of steam.")
print(f"   Rule of thumb: {d_temp:.0f} °C / 20 = {d_temp/20:.1f} % efficiency lost.")
print(f"   Measured from the fuel figures:      {d_fuel*100:.1f} %")

# how fast is it moving?
trend = q(f"""
SELECT   strftime('%W', r.ts) AS week, AVG(r.stack_temp_c) AS stack_c
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
WHERE    b.code = '{worst}'
GROUP BY week
ORDER BY week
""")
rise = trend["stack_c"].iloc[-1] - trend["stack_c"].iloc[0]
weeks = len(trend) - 1

print(f"\n2) THE TREND")
print(f"   {worst} climbed {rise:.1f} °C over {weeks} weeks — about {rise/weeks:.1f} °C a week.")
print(f"   At that rate it passes 200 °C in roughly "
      f"{(200 - trend['stack_c'].iloc[-1]) / (rise/weeks):.0f} more weeks.")

print("\nTwo instruments, two tables, one story — and a rate of change.")
print("That is what evidence looks like: the tubes are fouling, and it is not finished yet.")

---
## What you learned today

1. **A table is a list of one kind of thing.** If you cannot say what one row is in four
   words, it should probably be two tables.
2. **Store every fact once** and point at it from everywhere else. That is why a database
   stays correct while a spreadsheet slowly drifts.
3. **`GROUP BY` is the one to remember.** Turning 540 rows into one number per boiler is
   what found the problem today.
4. **Choose the group deliberately.** Grouped by boiler, one unit is bad. Grouped by *week*,
   it is getting *worse* — a different finding, from the same rows, because you asked a
   different question.
5. **Two independent measurements beat one.** The thermocouple and the gas meter do not talk
   to each other. When they agree, argument stops.
6. **The computer counts, you interpret.** SQL gave you 183 °C and 3.145 GJ per tonne. Only
   an engineer can say that means the tubes are fouling.

### Take it further — no answers provided

1. **Rolling average.** Daily numbers are noisy. Smooth them with a window function:
   `AVG(steam_t) OVER (PARTITION BY boiler_id ORDER BY log_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`.
2. **Missing data.** Every boiler should have a water test each week. Use a `LEFT JOIN` from
   `daily_logs` to `water_tests` to find the boiler-days with no test — and see what `NULL` does
   to your averages.
3. **The worst day.** For each boiler, find the single day with the highest gas per tonne, and
   show it next to that day's average stack temperature.
4. **Money.** Gas costs about ฿300 per GJ. How much did the fouling on the worst boiler cost in
   April alone? Write it as one query.
5. **When do we shut down?** Fit the weekly trend by hand and predict the week the worst boiler
   crosses 200 °C. Then argue for or against cleaning it at the next planned outage.
6. **Your own question.** Ask the database something nobody set you, and check whether the
   answer would survive someone disagreeing with it.

### Keeping your work

* **Your answers** are already saved if you did *File → Save a copy in Drive* at the start.
  If you did not, do it now — **File → Save a copy in Drive**.
* **The database file** is rebuilt by the first cell every time, so you never need to keep it.
  But if you would like a copy to open in DB Browser for SQLite at home, run the cell below.

In [ ]:
# Optional: download boilerhouse.db to your own computer (Colab only).
try:
    from google.colab import files
    files.download("boilerhouse.db")
except ImportError:
    print("Not running in Colab — the file is already in this folder:",
          os.path.abspath("boilerhouse.db"))

In [ ]:
con.close()
print("Done. Try asking the database a question nobody set you.")